# RPOE-X — GRPO Training Notebook

**Environment:** Rotary Parking Optimization — HITEC City, Hyderabad  
**Algorithm:** GRPO (Group Relative Policy Optimization) via HuggingFace TRL + Unsloth  
**Hardware:** NVIDIA GPU (Google Colab T4/A100 recommended)

### Agent Architecture
| Agent | Role | Learns |
|---|---|---|
| Orchestrator | Routes car to a zone (0–4) | Which zones have cars waiting |
| Zone Agent | Picks wheel within zone | Which wheels have free slots |

**Goal:** Task 1 service rate: ~0.25 (untrained) → 0.70+ (fine-tuned)

## 1. Install Dependencies

In [ ]:
%%capture
import os

os.system("pip install openenv-core")
os.system("pip install 'git+https://huggingface.co/spaces/bharavivillu/rpoe-x'")
os.system("pip install 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'")
os.system("pip install trl transformers accelerate peft datasets pydantic numpy matplotlib")
print("Done")

## 2. Imports & Model

In [ ]:
import os, sys, json, re, random, shutil
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset

sys.path.insert(0, os.path.abspath("."))

MODEL_ID       = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_SEQ_LENGTH = 512
LORA_RANK      = 16

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_RANK,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = 16,
    use_gradient_checkpointing = "unsloth",
)
print(f"Model : {MODEL_ID}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 3. Environment & Greedy Baselines

In [ ]:
from rpoe_x import ParkingEnv, ParkingAction

# Connect to HF Space:
# parking_env = ParkingEnv.from_env("bharavivillu/rpoe-x")
# Connect to local server:
# parking_env = ParkingEnv(base_url="http://localhost:8000")

from models import OrchestratorAction, ZoneAction, OrchestratorObs, ZoneObs, CarState
from server.env import RPOEXEnv, _open_score, ZONES, WHEEL_SIZE
from tasks.graders import greedy_orchestrator, greedy_zone

env  = RPOEXEnv(seed=42, max_steps=10)
obs  = env.reset()
obs2 = env.step(OrchestratorAction(action="route_to_zone", zone_id=2),
                ZoneAction(action="assign_to_wheel", wheel_id=0))
assert obs2.step == 1
print("Environment OK")

## 4. System Prompts for Both Agents

In [ ]:
ORCH_SYSTEM = """You are an orchestrator agent for a rotary parking system in HITEC City, Hyderabad.
Each step you route ONE car to ONE zone. Routing to an empty zone wastes the step.
RULE: Route to a zone with cars waiting (zone_queue_lengths > 0). Prefer the busiest zone.
If no zone has cars, use zone 2 (Hitech Metro — largest buffer).
Zones: 0=Cyber Towers, 1=Inorbit Mall, 2=Hitech Metro, 3=Mindspace, 4=Kondapur.
Respond ONLY with: {"zone_id": <int 0-4>} /no_think"""

ZONE_SYSTEM = """You are a zone agent for a rotary parking system.
Assign the incoming car to the best wheel in your zone.
RULE: Never pick a full wheel (occupancy=1.0 or queue_length=12). Pick the least-occupied wheel.
Respond ONLY with: {"wheel_id": <int>} /no_think"""


def orch_user_msg(obs):
    return (
        f"zone_occupancy: {[round(x,2) for x in obs.zone_occupancy]}\n"
        f"zone_queue_lengths: {obs.zone_queue_lengths}\n"
        f"zone_avg_wait: {[round(x,1) for x in obs.zone_avg_wait]}\n"
        f"arrival_rate_ema: {[round(x,3) for x in obs.arrival_rate_ema]}\n"
        f"time_of_day: {obs.time_of_day:.3f}  step: {obs.step}\n"
        f"Which zone_id (0-4) should the next car be routed to?"
    )


def zone_user_msg(obs):
    n = len(obs.wheel_occupancy)
    return (
        f"zone_id: {obs.zone_id}\n"
        f"wheel_occupancy: {[round(x,2) for x in obs.wheel_occupancy]}\n"
        f"wheel_queue_lengths: {obs.wheel_queue_lengths}\n"
        f"est_rotation_cost: {obs.est_rotation_cost}\n"
        f"time_of_day: {obs.time_of_day:.3f}  step: {obs.step}\n"
        f"Which wheel_id (0-{n-1}) should the car be assigned to?"
    )

print("Prompts defined")

## 5. Evaluate BEFORE Training

In [ ]:
import torch
from unsloth import FastLanguageModel


def generate_action(messages, max_new_tokens=24):
    FastLanguageModel.for_inference(model)
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()


def llm_orchestrator(obs):
    messages = [{"role": "system", "content": ORCH_SYSTEM},
                {"role": "user",   "content": orch_user_msg(obs)}]
    try:
        zone_id = int(json.loads(generate_action(messages))["zone_id"])
        assert 0 <= zone_id <= 4
        return OrchestratorAction(action="route_to_zone", zone_id=zone_id)
    except Exception:
        return greedy_orchestrator(obs)


def llm_zone_agent(obs):
    n = len(obs.wheel_occupancy)
    messages = [{"role": "system", "content": ZONE_SYSTEM},
                {"role": "user",   "content": zone_user_msg(obs)}]
    try:
        wheel_id = int(json.loads(generate_action(messages))["wheel_id"])
        assert 0 <= wheel_id < n
        return ZoneAction(action="assign_to_wheel", wheel_id=wheel_id)
    except Exception:
        return greedy_zone(obs)


def _run_episode(orch_fn, zone_fn, seed=42, max_steps=400, lambda_override=None):
    env = RPOEXEnv(seed=seed, max_steps=max_steps, lambda_override=lambda_override)
    obs = env.reset(seed=seed)
    while not obs.done:
        orch_action = orch_fn(obs)
        zone_obs    = env.get_zone_obs(orch_action.zone_id)
        zone_action = zone_fn(zone_obs)
        obs         = env.step(orch_action, zone_action)
    arrived = env._parked + env._overflowed
    return {"service_rate": _open_score(env._parked / max(1, arrived)),
            "parked": env._parked, "overflowed": env._overflowed}


def run_episode_llm(seed=42, max_steps=400, lambda_override=None):
    return _run_episode(llm_orchestrator, llm_zone_agent,
                        seed=seed, max_steps=max_steps, lambda_override=lambda_override)


def run_episode_greedy(seed=42, max_steps=400, lambda_override=None):
    return _run_episode(greedy_orchestrator, greedy_zone,
                        seed=seed, max_steps=max_steps, lambda_override=lambda_override)["service_rate"]


print("Evaluating BEFORE training (task1 seed=42, 200 steps)...")
greedy_score = run_episode_greedy(seed=42, max_steps=200, lambda_override=0.05)
before       = run_episode_llm(seed=42, max_steps=200, lambda_override=0.05)
print(f"Greedy baseline:        {greedy_score:.4f}")
print(f"Base model (untrained): {before['service_rate']:.4f}  "
      f"(parked={before['parked']} overflowed={before['overflowed']})")

## 6. Generate Training Dataset

In [ ]:
def build_dataset(n_episodes=30, max_steps=400):
    rows = []
    for ep in range(n_episodes):
        env = RPOEXEnv(seed=ep * 7 + 1, max_steps=max_steps)
        obs = env.reset(seed=ep * 7 + 1)
        while not obs.done:
            if any(q > 0 for q in obs.zone_queue_lengths):
                rows.append({
                    "prompt":             [{"role": "system", "content": ORCH_SYSTEM},
                                           {"role": "user",   "content": orch_user_msg(obs)}],
                    "agent_role":         "orchestrator",
                    "zone_queue_lengths": list(obs.zone_queue_lengths),
                    "wheel_occupancy":    [],
                    "n_wheels":           0,
                })
            orch_action = greedy_orchestrator(obs)
            zone_obs    = env.get_zone_obs(orch_action.zone_id)
            rows.append({
                "prompt":             [{"role": "system", "content": ZONE_SYSTEM},
                                       {"role": "user",   "content": zone_user_msg(zone_obs)}],
                "agent_role":         "zone",
                "zone_queue_lengths": [],
                "wheel_occupancy":    list(zone_obs.wheel_occupancy),
                "n_wheels":           len(zone_obs.wheel_occupancy),
            })
            zone_action = greedy_zone(zone_obs)
            obs = env.step(orch_action, zone_action)
    random.shuffle(rows)
    return Dataset.from_list(rows)


dataset = build_dataset(n_episodes=30)
orch_n  = sum(1 for r in dataset if r["agent_role"] == "orchestrator")
print(f"Dataset: {len(dataset)} rows  (orchestrator={orch_n} zone={len(dataset)-orch_n})")

## 7. Reward Functions

| Function | Applies to | Rewards |
|---|---|---|
| `format_reward` | Both | Valid JSON with correct key |
| `routing_reward` | Orchestrator | Routing to zones with queued cars |
| `wheel_reward` | Zone agent | Picking non-full wheels |

In [ ]:
def _parse(completion):
    try:
        text = (completion[0]["content"] if isinstance(completion, list) else completion).strip()
        text = re.sub(r"```.*?```", "", text, flags=re.DOTALL).strip()
        return json.loads(text)
    except Exception:
        return None


def format_reward(completions, agent_role, **kwargs):
    rewards = []
    for comp, role in zip(completions, agent_role):
        p = _parse(comp)
        if p is None:
            rewards.append(-1.0)
        elif role == "orchestrator" and "zone_id" in p and 0 <= int(p["zone_id"]) <= 4:
            rewards.append(0.5)
        elif role == "zone" and "wheel_id" in p and int(p["wheel_id"]) >= 0:
            rewards.append(0.5)
        else:
            rewards.append(-1.0)
    return rewards


def routing_reward(completions, agent_role, zone_queue_lengths, **kwargs):
    rewards = []
    for comp, role, ql in zip(completions, agent_role, zone_queue_lengths):
        if role != "orchestrator" or not ql:
            rewards.append(0.0)
            continue
        p = _parse(comp)
        if p is None or "zone_id" not in p:
            rewards.append(-0.5)
            continue
        zone_id = int(p["zone_id"])
        total   = sum(ql)
        if not (0 <= zone_id <= 4):
            rewards.append(-1.0)
        elif total == 0:
            rewards.append(0.0)
        elif ql[zone_id] == 0:
            rewards.append(-0.5)
        else:
            rewards.append(min(1.0, float(ql[zone_id]) / total + 0.3))
    return rewards


def wheel_reward(completions, agent_role, wheel_occupancy, n_wheels, **kwargs):
    rewards = []
    for comp, role, occ, nw in zip(completions, agent_role, wheel_occupancy, n_wheels):
        if role != "zone" or not occ:
            rewards.append(0.0)
            continue
        p = _parse(comp)
        if p is None or "wheel_id" not in p:
            rewards.append(-0.5)
            continue
        wid = int(p["wheel_id"])
        if not (0 <= wid < nw):
            rewards.append(-1.0)
        elif occ[wid] >= 1.0:
            rewards.append(-0.5)
        else:
            rewards.append(1.0 - occ[wid])
    return rewards


print("Reward functions defined")

## 8. Training (Unsloth + GRPO)

In [ ]:
if os.path.exists("/tmp/rpoe_x_grpo"):
    shutil.rmtree("/tmp/rpoe_x_grpo")

from trl import GRPOTrainer, GRPOConfig
from unsloth import PatchDPOTrainer
PatchDPOTrainer()
FastLanguageModel.for_training(model)

grpo_config = GRPOConfig(
    output_dir                  = "/tmp/rpoe_x_grpo",
    learning_rate               = 5e-6,
    lr_scheduler_type           = "cosine",
    max_steps                   = 200,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    num_generations             = 4,
    max_prompt_length           = 384,
    max_completion_length       = 32,
    temperature                 = 0.8,
    logging_steps               = 10,
    save_strategy               = "no",
    report_to                   = "none",
    fp16                        = True,
)
trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [format_reward, routing_reward, wheel_reward],
    args             = grpo_config,
    train_dataset    = dataset,
)
print(f"GRPO: {grpo_config.max_steps} steps, {grpo_config.num_generations} generations/prompt")
train_result = trainer.train()
print(f"Done. steps={train_result.global_step}  {train_result.metrics}")

## 9. Evaluate AFTER Training

In [ ]:
print("Evaluating AFTER training (task1 seed=42, 200 steps)...")
after = run_episode_llm(seed=42, max_steps=200, lambda_override=0.05)

print()
print("=" * 52)
print(f"{'Agent':<28} {'Service Rate':>12}")
print("-" * 52)
print(f"{'Greedy baseline':<28} {greedy_score:>12.4f}")
print(f"{'Base model (untrained)':<28} {before['service_rate']:>12.4f}")
print(f"{'Fine-tuned (GRPO)':<28} {after['service_rate']:>12.4f}")
print("=" * 52)
delta = after["service_rate"] - before["service_rate"]
pct   = delta / max(before["service_rate"], 0.001) * 100
print(f"Improvement over base model: {delta:+.4f}  ({pct:+.1f}%)")

## 10. Training Curve + Before/After Plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f"RPOE-X — GRPO Training  [{MODEL_ID}]", fontsize=13, fontweight="bold")

logs    = trainer.state.log_history
steps   = [l["step"]   for l in logs if "reward" in l]
rewards = [l["reward"] for l in logs if "reward" in l]
ax1.plot(steps, rewards, color="steelblue", linewidth=1.2, alpha=0.6, label="Per step")
if len(rewards) >= 10:
    sm = np.convolve(rewards, np.ones(10) / 10, mode="valid")
    ax1.plot(steps[9:], sm, color="steelblue", linewidth=2.5, label="Smoothed")
ax1.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax1.set_xlabel("Training Step")
ax1.set_ylabel("GRPO Reward")
ax1.set_title("Reward During Training")
ax1.legend()
ax1.grid(alpha=0.3)

labels = ["Greedy", "Base model\n(untrained)", "Fine-tuned\n(GRPO)"]
values = [greedy_score, before["service_rate"], after["service_rate"]]
colors = ["#e74c3c", "#f39c12", "#2ecc71"]
bars   = ax2.bar(labels, values, color=colors, width=0.5, edgecolor="black", linewidth=0.8)
ax2.axhline(0.50, color="navy", linestyle="--", linewidth=1.5, label="Pass threshold (0.50)")
for bar, v in zip(bars, values):
    ax2.text(bar.get_x() + bar.get_width() / 2, v + 0.01, f"{v:.3f}",
             ha="center", va="bottom", fontweight="bold", fontsize=11)
ax2.set_ylim(0, 1.05)
ax2.set_ylabel("Task 1 Service Rate")
ax2.set_title("Before vs After Training")
ax2.legend()
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("training_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: training_curve.png")

## 11. Save Trained Model

In [ ]:
SAVE_PATH = "trained_rpoe_x_agent"
model.save_pretrained_merged(SAVE_PATH, tokenizer, save_method="merged_16bit")
print(f"Merged model saved to {SAVE_PATH}/")
# Push to HF Hub (uncomment when ready):
# model.push_to_hub("bharavivillu/rpoe-x-qwen-0.5b-grpo")
# tokenizer.push_to_hub("bharavivillu/rpoe-x-qwen-0.5b-grpo")
print(f"\nTo run inference: MODEL_PATH={SAVE_PATH} python inference.py")